In [9]:
import os, sys, time, json, math, random, uuid, re
from typing import List, Dict, Any, Tuple, Iterable
from collections import Counter, defaultdict
from pathlib import Path

SEED = 42
random.seed(SEED)

# =================== Config rápida ===================
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o")
EMBED_MODEL  = os.getenv("EMBED_MODEL", "text-embedding-3-small")
BP_MODE      = os.getenv("BP_MODE", "memory")  # greedy | memory | sliding_window

BP_VARIANTS_PER_CALL   = int(os.getenv("BP_VARIANTS_PER_CALL", "4"))
BP_MAX_CALLS_PER_ITEM  = int(os.getenv("BP_MAX_CALLS_PER_ITEM", "3"))
MIN_JACCARD_FOR_LABEL  = float(os.getenv("MIN_JACCARD_FOR_LABEL", "0.25"))
MAX_JACCARD_FOR_DIVERSITY = float(os.getenv("MAX_JACCARD_FOR_DIVERSITY", "0.92"))


INPUT_JSONL  = os.getenv("INPUT_JSONL",  "merged_masked_unionv2_new/merged_masked_unionv2_new_train.jsonl")
OUTPUT_JSONL = os.getenv("OUTPUT_JSONL", "merged_masked_unionv2_new/merged_masked_unionv2_new_train_tga_rag.jsonl")

# Sliding window
SW_WINDOW_SIZE = int(os.getenv("SW_WINDOW_SIZE", "3"))   # n palabras por ventana
SW_STRIDE      = int(os.getenv("SW_STRIDE", "3"))        # salto (usa n para no solapar)

# Memoria / RAG
USE_EMBEDDINGS         = os.getenv("USE_EMBEDDINGS", "1") not in ("0","false","False")
SIM_THRESHOLD          = float(os.getenv("SIM_THRESHOLD", "0.88"))  # coseno
MAX_NEG_EXAMPLES       = int(os.getenv("MAX_NEG_EXAMPLES", "4"))    # p.ej. 2-6
RAG_NEIGHBORS_PER_CAT  = int(os.getenv("RAG_NEIGHBORS_PER_CAT", "64"))

RETRIES      = 2
RETRY_SLEEP  = 1.0
BP_RETRIES   = 6
SEED_JITTER  = True  # cambia seed por llamada para mayor varianza

# =================== Taxonomía (tal cual) ===================
TAXONOMY = [
    "GENDER_WOMEN","GENDER_MEN","GENDER_NONBINARY","GENDER_IDENTITY_TRANS","GENDER_IDENTITY_OTHER",
    "RACE_BLACK","RACE_WHITE","RACE_OTHER",
    "ETH_ASIAN","ETH_LATINO","ETH_ARAB","ETH_JEWISH","ETH_ROMA","ETH_OTHER",
    "SEXUAL_ORIENTATION_GAY","SEXUAL_ORIENTATION_LESBIAN","SEXUAL_ORIENTATION_BI","SEXUAL_ORIENTATION_OTHER",
    "RELIGION_ISLAM","RELIGION_CHRIST","RELIGION_JUDAISM","RELIGION_HINDU","RELIGION_SIKH","RELIGION_BUDDHIST","RELIGION_ATHEIST","RELIGION_OTHER",
    "NATIONAL_ORIGIN_IMMIGRANTS","NATIONAL_ORIGIN_REFUGEES","NATIONAL_ORIGIN_OTHER",
    "DISABILITY_PHYSICAL","DISABILITY_MENTAL","DISABILITY_DEVELOPMENTAL","DISABILITY_OTHER",
    "AGE_YOUTH","AGE_MIDDLE","AGE_ELDERLY","AGE_OTHER",
]
TAXO_STR = ", ".join(TAXONOMY)

# =================== Utilidades ===================
def read_api_key_from_file(path: str = "API_KEY") -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read().strip()

def ensure_api_key():
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = read_api_key_from_file()

def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: str, rows: List[Dict[str, Any]]):
    Path(os.path.dirname(path) or ".").mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_valid_row(row: Dict[str, Any]) -> bool:
    cat = row.get("predicted_hate_category", None)
    return (cat is not None) and (str(cat).lower() != "nan") and bool(row.get("text","").strip())

# =================== OpenAI ===================
from openai import OpenAI
client = None

def init_openai():
    global client
    ensure_api_key()
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# =================== Regex y tokens ===================
TOKEN_RE      = re.compile(r"\[(SLUR|TARGET|URL|MENTION|HASHTAG):([A-Z0-9_]+|\w+)?\]")
STRICT_TOKEN  = re.compile(r"\[(SLUR|TARGET):([A-Z0-9_]+)\]")  # para validación fuerte
WORD_RE       = re.compile(r"[^\W_]+(?:'[^\W_]+)?(?:-[^\W_]+)?", flags=re.UNICODE)
FENCE_RE      = re.compile(r"^```(?:\w+)?\s*([\s\S]*?)\s*```$", re.IGNORECASE)

def _strip_fences(s: str) -> str:
    m = FENCE_RE.match(s.strip())
    return m.group(1).strip() if m else s.strip()

def contains_all(orig_tokens: List[str], s: str) -> bool:
    return all(tok in s for tok in orig_tokens)

# =================== Memoria / similitud ===================
def normalize_text_for_jaccard(s: str) -> List[str]:
    s = re.sub(r"\s+", " ", s.lower()).strip()
    # 3-gramas de caracteres (rápido, robusto)
    return [s[i:i+3] for i in range(max(1, len(s)-2))]

def jaccard_sim(a: str, b: str) -> float:
    A = set(normalize_text_for_jaccard(a))
    B = set(normalize_text_for_jaccard(b))
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

class MemoryStore:
    """Memoria por categoría con embeddings opcionales + textos crudos para anti-duplicación y RAG negativo."""
    def __init__(self, use_embeddings: bool = True, embed_model: str = "text-embedding-3-small"):
        self.use_embeddings = use_embeddings
        self.embed_model = embed_model
        self.by_cat_texts: Dict[str, List[str]] = defaultdict(list)
        self.by_cat_embs:  Dict[str, List[List[float]]] = defaultdict(list)

    def _embed_batch(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []
        resp = client.embeddings.create(model=self.embed_model, input=texts)
        return [d.embedding for d in resp.data]

    @staticmethod
    def _cos_sim(u: List[float], v: List[float]) -> float:
        import math
        num = sum(x*y for x,y in zip(u,v))
        du  = math.sqrt(sum(x*x for x in u))
        dv  = math.sqrt(sum(y*y for y in v))
        if du == 0 or dv == 0: return 0.0
        return num/(du*dv)

    def add(self, cat: str, texts: List[str]):
        if not texts: return
        self.by_cat_texts[cat].extend(texts)
        if self.use_embeddings:
            embs = self._embed_batch(texts)
            self.by_cat_embs[cat].extend(embs)

    def nearest(self, cat: str, query: str, k: int = 8) -> List[Tuple[str, float]]:
        """Devuelve (texto, similitud) de los k más cercanos por coseno (o Jaccard si no hay embeddings)."""
        corpus = self.by_cat_texts.get(cat, [])
        if not corpus: return []
        if self.use_embeddings and self.by_cat_embs.get(cat):
            q = self._embed_batch([query])[0]
            sims = [(t, self._cos_sim(q, e)) for t, e in zip(corpus, self.by_cat_embs[cat])]
        else:
            sims = [(t, jaccard_sim(query, t)) for t in corpus]
        sims.sort(key=lambda x: x[1], reverse=True)
        return sims[:k]

    def too_similar(self, cat: str, query: str, threshold: float) -> bool:
        if not self.by_cat_texts.get(cat): return False
        return any(sim >= threshold for _, sim in self.nearest(cat, query, k=1))

# =================== Prompts ===================
SYSTEM_PROMPT_GREEDY = (
    "You are rewriting short toxic social media posts (tweets). "
    "Given a MASKED input, you will produce new toxic tweets that keep the SAME toxicity, stance, and targets.\n"
    "STYLE RULES:\n"
    "- Write like a real toxic tweet: short, informal, sometimes slang.\n"
    "- Prefer simple, everyday words. Avoid formal or academic phrases "
    "  like 'therefore', 'it would be wise', 'as a consequence', etc.\n"
    "- Max 1–2 sentences per variant.\n"
    "HARD RULES:\n"
    "1) Preserve EVERY bracketed token EXACTLY as written (verbatim): [SLUR:CODE], [TARGET:CODE], [MENTION], [URL], [HASHTAG:tag]. You may reorder them but do NOT change inside.\n"
    "2) Do NOT introduce or reveal explicit slurs; only keep the placeholders.\n"
    "3) Vary wording and structure A LOT across variants: sometimes very short, sometimes with extra context, sometimes changing clause order.\n"
    "4) Do NOT add long explanations around [URL] or [MENTION]; keep them in-line as in tweets.\n"
    "5) You MUST respond with a single JSON object of the form "
    "{\"variants\": [\"...\", \"...\", ...]} containing N_VARIANTS different paraphrases.\n"
    "6) Do not include any extra commentary, prose, or code fences outside the JSON."
)

SYSTEM_PROMPT_MEMORY = (
    "You are rewriting short toxic social media posts (tweets). "
    "Given a MASKED input, you will produce new toxic tweets that keep the SAME toxicity, stance, and targets.\n"
    "STYLE RULES:\n"
    "- Write like a real toxic tweet: short, informal, sometimes slang.\n"
    "- Prefer simple, everyday words. Avoid formal or academic phrases "
    "  like 'therefore', 'it would be wise', 'as a consequence', etc.\n"
    "- Max 1–2 sentences per variant.\n"
    "HARD RULES:\n"
    "1) Preserve EVERY bracketed token EXACTLY as written (verbatim): [SLUR:CODE], [TARGET:CODE], [MENTION], [URL], [HASHTAG:tag]. You may reorder them but do NOT change inside.\n"
    "2) Do NOT introduce or reveal explicit slurs; only keep the placeholders.\n"
    "3) Vary wording and structure A LOT across variants: sometimes very short, sometimes with extra context, sometimes changing clause order.\n"
    "4) Do NOT add long explanations around [URL] or [MENTION]; keep them in-line as in tweets.\n"
    "5) You MUST respond with a single JSON object of the form "
    "{\"variants\": [\"...\", \"...\", ...]} containing N_VARIANTS different paraphrases.\n"
    "6) Do not include any extra commentary, prose, or code fences outside the JSON."
)


def paraphrase_memory_multi(masked_text: str, cat: str, lang: str, mem: MemoryStore, n_variants: int) -> List[str]:
    """Genera hasta n_variants candidatas usando memoria negativa y salida JSON."""
    negs = mem.nearest(cat, masked_text, k=MAX_NEG_EXAMPLES)
    neg_texts = [t for (t, s) in negs]
    neg_block = "\n---\n".join(neg_texts) if neg_texts else "(none)"

    user_prompt = (
        f"N_VARIANTS: {n_variants}\n"
        f"LANG: {lang}\n"
        f"CATEGORY: {cat}\n"
        "MASKED INPUT:\n"
        f"{masked_text}\n\n"
        "NEGATIVE EXAMPLES (avoid similar wording):\n"
        f"{neg_block}\n\n"
        "Return a JSON object: {\"variants\": [\"paraphrase 1\", \"paraphrase 2\", ...]}."
    )

    raw = call_with_retries(SYSTEM_PROMPT_MEMORY, user_prompt, max_tokens=512)
    s = _strip_fences(raw)

    # Intento de parseo robusto
    variants: List[str] = []
    try:
        data = json.loads(s)
        vlist = data.get("variants", [])
        if isinstance(vlist, list):
            variants = [str(v).strip() for v in vlist if str(v).strip()]
    except Exception:
        # fallback: si no es JSON válido, tratamos toda la salida como una sola variante
        if s.strip():
            variants = [s.strip()]

    return variants

def paraphrase_memory(masked_text: str, cat: str, lang: str, mem: MemoryStore) -> str:
    """Wrapper compatible: devuelve solo una variante (la primera válida)."""
    vs = paraphrase_memory_multi(masked_text, cat, lang, mem, n_variants=1)
    return vs[0] if vs else masked_text


def jitter_hparams(base_temp=0.95, base_top_p=0.95, base_pp=0.45, base_fp=0.35):
    """Pequeño jitter para variedad entre llamadas."""
    # rangos suaves pero efectivos
    temp = min(1.5, max(0.6, random.gauss(base_temp, 0.15)))
    top_p = min(1.0, max(0.85, random.gauss(base_top_p, 0.05)))
    pres  = min(1.2, max(0.0, random.gauss(base_pp, 0.15)))
    freq  = min(1.2, max(0.0, random.gauss(base_fp, 0.15)))
    seed  = random.randint(1, 10_000_000) if SEED_JITTER else SEED
    return dict(temperature=temp, top_p=top_p, presence_penalty=pres, frequency_penalty=freq, seed=seed)

# =================== Llamadas LLM ===================
def call_chat_once(system_prompt: str, user_prompt: str, max_tokens: int = 300, **kw) -> str:
    hp = jitter_hparams()
    hp.update(kw)
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=max_tokens,
        **hp
    )
    out = (resp.choices[0].message.content or "").strip()
    return _strip_fences(out)

def call_with_retries(system_prompt: str, user_prompt: str, retries: int = BP_RETRIES, **kw) -> str:
    backoff = 1.0
    last_e = None
    for attempt in range(retries + 1):
        try:
            out = call_chat_once(system_prompt, user_prompt, **kw)
            if not out:
                raise ValueError("Empty output")
            return out
        except Exception as e:
            last_e = e
            if attempt >= retries:
                raise last_e
            time.sleep(backoff + random.random() * 0.25)
            backoff = min(16.0, backoff * 2)
    raise last_e or RuntimeError("Failed after retries")

# =================== Estrategias ===================
def paraphrase_greedy(masked_text: str, cat: str, lang: str) -> str:
    user_prompt = f"LANG: {lang}\nCATEGORY: {cat}\nMASKED INPUT:\n{masked_text}\n\nRemember: output ONLY the paraphrase."
    return call_with_retries(SYSTEM_PROMPT_GREEDY, user_prompt, max_tokens=300)

def paraphrase_memory(masked_text: str, cat: str, lang: str, mem: MemoryStore) -> str:
    negs = mem.nearest(cat, masked_text, k=MAX_NEG_EXAMPLES)
    neg_texts = [t for (t, s) in negs]
    neg_block = "\n---\n".join(neg_texts) if neg_texts else "(none)"
    user_prompt = (
        f"LANG: {lang}\nCATEGORY: {cat}\n"
        "MASKED INPUT:\n"
        f"{masked_text}\n\n"
        "NEGATIVE EXAMPLES (avoid similar wording):\n"
        f"{neg_block}\n\n"
        "OUTPUT:"
    )
    return call_with_retries(SYSTEM_PROMPT_MEMORY, user_prompt, max_tokens=320)

def rewrite_window_infill(original_with_hole: str, n_words: int) -> str:
    """
    Pide al LLM que SOLO rellene el hueco HOLE_TOKEN con ~n_words palabras.
    Devuelve el texto que sustituirá al hueco.
    """
    user_prompt = (
        f"N_WORDS: {n_words}\n"
        "ORIGINAL (with HOLE):\n"
        f"{original_with_hole}\n\n"
        f"Replace {HOLE_TOKEN} with ~N_WORDS words. Output ONLY the replacement text."
    )
    out = call_with_retries(SYSTEM_PROMPT_SW_INFILL, user_prompt, max_tokens=max(64, n_words*4))
    # limpieza ligera
    out = _strip_fences(out).strip().strip('"').strip("'").strip()
    # fuerza longitud aproximada
    words = WORD_RE.findall(out)
    if not (0.8*n_words <= len(words) <= 1.2*n_words):
        strong_user = user_prompt + f"\n\nEnsure around {n_words} words (±20%). Output ONLY the replacement."
        out = call_with_retries(SYSTEM_PROMPT_SW_INFILL, strong_user, max_tokens=max(64, n_words*4))
        out = _strip_fences(out).strip().strip('"').strip("'").strip()
    # no permitir nuevos tokens entre corchetes en el relleno
    if re.search(r"\[[^\]]+\]", out):
        out = re.sub(r"\[[^\]]+\]", "", out).strip()
    return out

# ========= Helpers =========
# Detecta tokens con corchetes (SLUR/TARGET/URL/MENTION/HASHTAG)
FULL_TOKEN_RE = re.compile(r"\[(?:SLUR|TARGET|URL|MENTION|HASHTAG):[^\]]+\]")

def _word_matches(s: str):
    # "palabras" separadas por espacios (no rompe tokens, porque tokens llevan corchetes)
    return list(re.finditer(r"\S+", s))

# ========= Sliding Window por INFILL =========
def sliding_window_obfuscate(full: str, n: int, stride: int) -> str:
    """
    INFILL por ventanas de 'n' palabras:
      - Selecciona ventanas SOLO en tramos sin tokens [SLUR:...] / [TARGET:...] / ...
      - Crea un hueco en la oración completa (HOLE_TOKEN) y pide al LLM que lo rellene ~n palabras.
      - Mantiene TODOS los tokens entre corchetes, sin tocarlos.
    Nota: el 'stride' se aplica en número de palabras dentro de cada tramo sin tokens.
    """
    if not isinstance(full, str) or not full.strip():
        return full

    # Validación: tokens originales (se deben preservar 1:1)
    orig_tokens = [m.group(0) for m in STRICT_TOKEN.finditer(full)]
    s = full
    step = stride if isinstance(stride, int) and stride > 0 else n

    idx = 0
    while True:
        # Busca siguiente token desde idx en la cadena ACTUAL
        m = FULL_TOKEN_RE.search(s, idx)
        chunk_start = idx
        chunk_end = m.start() if m else len(s)

        # Procesa el tramo sin tokens [chunk_start:chunk_end] con ventanas
        start_word_idx = 0
        while True:
            chunk = s[chunk_start:chunk_end]
            word_matches = _word_matches(chunk)
            if not word_matches or start_word_idx >= len(word_matches):
                break

            end_word_idx = min(len(word_matches), start_word_idx + n)
            i_rel = word_matches[start_word_idx].start()
            j_rel = word_matches[end_word_idx - 1].end()

            i_abs = chunk_start + i_rel
            j_abs = chunk_start + j_rel

            # Construye la ORACIÓN COMPLETA con HUECO
            original_with_hole = s[:i_abs] + HOLE_TOKEN + s[j_abs:]
            n_words_win = end_word_idx - start_word_idx

            replacement = rewrite_window_infill(original_with_hole, n_words=n_words_win)
            replacement = re.sub(r"\s+", " ", replacement).strip()
            # Evita que el relleno contenga nuevos tokens en corchetes
            if "[" in replacement and "]" in replacement:
                replacement = re.sub(r"\[[^\]]+\]", "", replacement).strip()

            # Aplica el relleno
            before = s[:i_abs]
            after  = s[j_abs:]
            s = before + replacement + after

            # Actualiza límites del tramo tras el cambio
            delta = len(replacement) - (j_abs - i_abs)
            chunk_end += delta  # el final del chunk se mueve con el reemplazo

            # Avanza el índice de palabras dentro del TRAMO (recalcular en la próxima iteración)
            start_word_idx += step

        # Mover 'idx' justo después del token (si lo hay)
        idx = chunk_end
        if m:
            # El token debería comenzar ahora en 'idx' exacto; confírmalo y sáltalo
            m2 = FULL_TOKEN_RE.match(s, idx)
            if m2:
                idx = m2.end()
            else:
                # si no cuadra exactamente por cambios, buscamos el siguiente desde idx
                m3 = FULL_TOKEN_RE.search(s, idx)
                if m3:
                    idx = m3.end()
                else:
                    # no quedan tokens
                    pass
        # Si no hay más tokens y hemos procesado hasta el final, salimos
        if not m and idx >= len(s):
            break

    # Validación final de tokens: deben coincidir exactamente
    final_tokens = [m.group(0) for m in STRICT_TOKEN.finditer(s)]
    if Counter(orig_tokens) != Counter(final_tokens):
        raise ValueError("sliding_window_obfuscate: los tokens en corchetes cambiaron (abort).")

    return s

# =================== Núcleo Balancing (como el tuyo) ===================
def compute_targets(by_cat: Dict[str, List[Dict[str, Any]]]) -> Tuple[int, Dict[str, int]]:
    counts = {c: len(v) for c, v in by_cat.items()}
    target_size = max(counts.values())
    needs = {c: max(0, target_size - n) for c, n in counts.items()}
    return target_size, needs

def distribute_k_per_item(n_need: int, n_items: int) -> List[int]:
    if n_need <= 0 or n_items == 0:
        return [0] * n_items
    base = n_need // n_items
    rem  = n_need % n_items
    ks = [base] * n_items
    for i in range(rem):
        ks[i] += 1
    random.shuffle(ks)
    return ks

# =================== Main ===================
def main():
    init_openai()
    rows = [r for r in read_jsonl(INPUT_JSONL) if is_valid_row(r)]
    rows = [r for r in rows if isinstance(r.get("text_masked"), str) and STRICT_TOKEN.search(r["text_masked"])]
    if not rows:
        print("No hay filas válidas con tokens [SLUR/TARGET].")
        return

    by_cat = defaultdict(list)
    for r in rows:
        by_cat[r["predicted_hate_category"]].append(r)

    target_size, needs = compute_targets(by_cat)
    print("Distribución inicial:", {c: len(v) for c, v in by_cat.items()})
    print("Objetivo por clase:", target_size)
    print("Necesidades:", needs)

    memory = MemoryStore(use_embeddings=USE_EMBEDDINGS, embed_model=EMBED_MODEL)

    # precarga memoria con los textos originales por clase (para anti-duplicar)
    for cat, items in by_cat.items():
        base_texts = [it["text_masked"] for it in items]
        base_texts = base_texts[:RAG_NEIGHBORS_PER_CAT]
        memory.add(cat, base_texts)

    augmented: List[Dict[str, Any]] = []

    ordered_cats = sorted(by_cat.keys(), key=lambda c: needs.get(c, 0), reverse=True)
    print("Orden de procesamiento (minor→major):",
          [(c, len(by_cat[c]), needs.get(c, 0)) for c in ordered_cats])

    for cat in ordered_cats:
        items = by_cat[cat]
        need  = needs.get(cat, 0)
        if need <= 0:
            print(f"[{cat}] ya está en objetivo ({len(items)})")
            continue

        ks = distribute_k_per_item(need, len(items))
        produced = 0
        random.shuffle(items)

        for row, k_i in zip(items, ks):
            if k_i <= 0:
                continue

            lang        = row.get("lang", "en")
            parent_id   = row.get("id")
            masked_orig = row["text_masked"]
            orig_tokens = [m.group(0) for m in STRICT_TOKEN.finditer(masked_orig)]

            print(f"\n[{cat}] seed id={parent_id} need={k_i}")
            print(f"  seed_masked: {masked_orig}")

            # --- modo greedy: dejamos casi igual que antes ---
            if BP_MODE == "greedy":
                masked_in = masked_orig
                for step in range(1, k_i + 1):
                    try:
                        masked_out = paraphrase_greedy(masked_in, cat, lang)
                        if not contains_all(orig_tokens, masked_out):
                            raise ValueError("Missing masked tokens")

                        new_row = dict(row)
                        new_row["id"] = f"{row.get('id','NA')}_bp_{uuid.uuid4().hex[:8]}"
                        new_row["text_masked"] = masked_out
                        new_row["groups_from_mask"] = sorted({m.group(2) for m in STRICT_TOKEN.finditer(masked_out)})
                        new_row["augmented"] = True
                        new_row["aug_method"] = f"broken_phone_llm_{BP_MODE}"
                        new_row["source_id"] = parent_id
                        new_row["bp_step"] = step

                        augmented.append(new_row)
                        produced += 1
                        print(f"  greedy step={step}: {masked_out}")

                        masked_in = masked_out
                        parent_id = new_row["id"]

                    except Exception as e:
                        sys.stderr.write(f"[WARN] greedy {cat} id={row.get('id')} step={step} err={e}\n")
                        continue
                continue  # siguiente fila

            # --- modo memory: nuevas llamadas multi-variantes ---
            if BP_MODE == "memory":
                remaining   = k_i
                bp_step     = 0
                masked_seed = masked_orig
                calls_used  = 0

                while remaining > 0 and calls_used < BP_MAX_CALLS_PER_ITEM:
                    calls_used += 1
                    n_variants = min(BP_VARIANTS_PER_CALL, max(1, remaining * 2))

                    try:
                        cand_list = paraphrase_memory_multi(
                            masked_seed, cat, lang, memory, n_variants=n_variants
                        )
                    except Exception as e:
                        sys.stderr.write(f"[WARN] memory_call {cat} id={row.get('id')} call={calls_used} err={e}\n")
                        break

                    if not cand_list:
                        break

                    for cand in cand_list:
                        if remaining <= 0:
                            break
                        cand = _strip_fences(cand).strip()
                        if not cand:
                            continue

                        # 1) comprobar que todos los tokens [SLUR/TARGET] siguen presentes
                        if not contains_all(orig_tokens, cand):
                            continue

                        # 2) rango de similitud por Jaccard vs el original (para no cambiar de intención)
                        jac_orig = jaccard_sim(masked_orig, cand)
                        if jac_orig < MIN_JACCARD_FOR_LABEL:
                            # demasiado diferente, probablemente cambia de significado
                            continue

                        # 3) evitar clones usando memoria (coseno o Jaccard según config)
                        if memory.too_similar(cat, cand, SIM_THRESHOLD):
                            continue

                        # 4) opcional: evitar parafraseos casi idénticos al original
                        if jaccard_sim(masked_orig, cand) > MAX_JACCARD_FOR_DIVERSITY:
                            continue

                        # si pasa todos los filtros, lo aceptamos
                        memory.add(cat, [cand])

                        new_row = dict(row)
                        bp_step += 1
                        new_row["id"] = f"{row.get('id','NA')}_bp_{uuid.uuid4().hex[:8]}"
                        new_row["text_masked"] = cand
                        new_row["groups_from_mask"] = sorted({m.group(2) for m in STRICT_TOKEN.finditer(cand)})
                        new_row["augmented"] = True
                        new_row["aug_method"] = f"broken_phone_llm_{BP_MODE}"
                        new_row["source_id"] = parent_id
                        new_row["bp_step"] = bp_step

                        augmented.append(new_row)
                        produced  += 1
                        remaining -= 1
                        print(f"  memory step={bp_step} (call={calls_used}): {cand}")

                        # efecto teléfono: la siguiente llamada parte de este candidato
                        masked_seed = cand
                        parent_id   = new_row["id"]

                continue  # siguiente fila

        print(f"[{cat}] añadidos={produced} → total esperado≈{len(items)+produced} (objetivo={target_size})")

    # mezcla y recorte suave por clase
    out_rows = rows + augmented
    final_by_cat = defaultdict(list)
    for r in out_rows:
        final_by_cat[r["predicted_hate_category"]].append(r)

    trimmed = []
    for c, lst in final_by_cat.items():
        if len(lst) > target_size:
            trimmed.extend(lst[:target_size])
        else:
            trimmed.extend(lst)

    write_jsonl(OUTPUT_JSONL, trimmed)
    final_counts = Counter(r["predicted_hate_category"] for r in trimmed)
    print("\nGuardado:", OUTPUT_JSONL)
    print("Distribución final:", dict(final_counts))
    print("Objetivo:", target_size)


In [10]:
if __name__ == "__main__":
    main()

Distribución inicial: {'hate_race': 476, 'hate_gender': 668, 'hate_sexuality': 215, 'hate_religion': 184, 'hate_disability': 18, 'hate_origin': 127}
Objetivo por clase: 668
Necesidades: {'hate_race': 192, 'hate_gender': 0, 'hate_sexuality': 453, 'hate_religion': 484, 'hate_disability': 650, 'hate_origin': 541}
Orden de procesamiento (minor→major): [('hate_disability', 18, 650), ('hate_origin', 127, 541), ('hate_religion', 184, 484), ('hate_sexuality', 215, 453), ('hate_race', 476, 192), ('hate_gender', 668, 0)]

[hate_disability] seed id=1114 need=36
  seed_masked: [MENTION] [MENTION] Deadass tho ,, buddy half past [SLUR:DISABILITY_OTHER] for that one 😂😭
  memory step=1 (call=1): Yo [MENTION] what was that? Dude's gone full [SLUR:DISABILITY_OTHER] 😂😭
  memory step=2 (call=1): Bruh, tell me why he's acting all [SLUR:DISABILITY_OTHER] now? 😂😭 [MENTION]
  memory step=3 (call=1): I can't with this! Man's out here looking [SLUR:DISABILITY_OTHER] for real 😂😭 [MENTION]
  memory step=4 (call=2

In [ ]:
import os, sys, time, json, math, random, uuid, re
from typing import List, Dict, Any, Tuple, Iterable
from collections import Counter, defaultdict
from pathlib import Path

# Instalar: pip install openai sentence-transformers
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)

# =================== Config rápida ===================
# DeepSeek V3 usa "deepseek-chat" (o deepseek-coder)
LLM_MODEL    = os.getenv("LLM_MODEL", "deepseek-chat")

# Usaremos un modelo local rápido para embeddings ya que DeepSeek no tiene endpoint de embeddings
EMBED_MODEL  = os.getenv("EMBED_MODEL", "all-MiniLM-L6-v2")

BP_MODE      = os.getenv("BP_MODE", "memory")  # greedy | memory | sliding_window

BP_VARIANTS_PER_CALL   = int(os.getenv("BP_VARIANTS_PER_CALL", "4"))
BP_MAX_CALLS_PER_ITEM  = int(os.getenv("BP_MAX_CALLS_PER_ITEM", "3"))
MIN_JACCARD_FOR_LABEL  = float(os.getenv("MIN_JACCARD_FOR_LABEL", "0.25"))
MAX_JACCARD_FOR_DIVERSITY = float(os.getenv("MAX_JACCARD_FOR_DIVERSITY", "0.92"))

INPUT_JSONL  = os.getenv("INPUT_JSONL",  "en_dataset_with_stop_words/en_dataset_with_stop_words_masked_train_low_regime_augmented.jsonl")
OUTPUT_JSONL = os.getenv("OUTPUT_JSONL", "en_dataset_with_stop_words/en_dataset_with_stop_words_masked_train_low_regime_augmentedv2.jsonl")

# Sliding window
SW_WINDOW_SIZE = int(os.getenv("SW_WINDOW_SIZE", "3"))
SW_STRIDE      = int(os.getenv("SW_STRIDE", "3"))

# Memoria / RAG
USE_EMBEDDINGS         = os.getenv("USE_EMBEDDINGS", "1") not in ("0","false","False")
SIM_THRESHOLD          = float(os.getenv("SIM_THRESHOLD", "0.88"))
MAX_NEG_EXAMPLES       = int(os.getenv("MAX_NEG_EXAMPLES", "4"))
RAG_NEIGHBORS_PER_CAT  = int(os.getenv("RAG_NEIGHBORS_PER_CAT", "64"))

RETRIES      = 2
RETRY_SLEEP  = 1.0
BP_RETRIES   = 6
SEED_JITTER  = True

# =================== Taxonomía ===================
TAXONOMY = [
    "GENDER_WOMEN","GENDER_MEN","GENDER_NONBINARY","GENDER_IDENTITY_TRANS","GENDER_IDENTITY_OTHER",
    "RACE_BLACK","RACE_WHITE","RACE_OTHER",
    "ETH_ASIAN","ETH_LATINO","ETH_ARAB","ETH_JEWISH","ETH_ROMA","ETH_OTHER",
    "SEXUAL_ORIENTATION_GAY","SEXUAL_ORIENTATION_LESBIAN","SEXUAL_ORIENTATION_BI","SEXUAL_ORIENTATION_OTHER",
    "RELIGION_ISLAM","RELIGION_CHRIST","RELIGION_JUDAISM","RELIGION_HINDU","RELIGION_SIKH","RELIGION_BUDDHIST","RELIGION_ATHEIST","RELIGION_OTHER",
    "NATIONAL_ORIGIN_IMMIGRANTS","NATIONAL_ORIGIN_REFUGEES","NATIONAL_ORIGIN_OTHER",
    "DISABILITY_PHYSICAL","DISABILITY_MENTAL","DISABILITY_DEVELOPMENTAL","DISABILITY_OTHER",
    "AGE_YOUTH","AGE_MIDDLE","AGE_ELDERLY","AGE_OTHER",
]
TAXO_STR = ", ".join(TAXONOMY)

# =================== Utilidades ===================
def read_api_key_from_file(path: str = "DAPI_KEY") -> str:
    """Lee la API Key del fichero indicado (por defecto DAPI_KEY)."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"ERROR: No se encontró el fichero '{path}' con la API Key.")
        sys.exit(1)

def ensure_api_key():
    if not os.getenv("DEEPSEEK_API_KEY"):
        os.environ["DEEPSEEK_API_KEY"] = read_api_key_from_file("DAPI_KEY")

def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: str, rows: List[Dict[str, Any]]):
    Path(os.path.dirname(path) or ".").mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_valid_row(row: Dict[str, Any]) -> bool:
    cat = row.get("predicted_hate_category", None)
    return (cat is not None) and (str(cat).lower() != "nan") and bool(row.get("text","").strip())

# =================== Cliente DeepSeek ===================
import os
from openai import OpenAI
client = None

def init_client():
    global client
    ensure_api_key()
    # Inicialización compatible con DeepSeek
    client = OpenAI(
        api_key=os.environ["DEEPSEEK_API_KEY"], 
        base_url="https://api.deepseek.com"
    )

# =================== Regex y tokens ===================
TOKEN_RE      = re.compile(r"\[(SLUR|TARGET|URL|MENTION|HASHTAG):([A-Z0-9_]+|\w+)?\]")
STRICT_TOKEN  = re.compile(r"\[(SLUR|TARGET):([A-Z0-9_]+)\]")
WORD_RE       = re.compile(r"[^\W_]+(?:'[^\W_]+)?(?:-[^\W_]+)?", flags=re.UNICODE)
FENCE_RE      = re.compile(r"^```(?:\w+)?\s*([\s\S]*?)\s*```$", re.IGNORECASE)

def _strip_fences(s: str) -> str:
    m = FENCE_RE.match(s.strip())
    return m.group(1).strip() if m else s.strip()

def contains_all(orig_tokens: List[str], s: str) -> bool:
    return all(tok in s for tok in orig_tokens)

# =================== Memoria / similitud ===================
def normalize_text_for_jaccard(s: str) -> List[str]:
    s = re.sub(r"\s+", " ", s.lower()).strip()
    return [s[i:i+3] for i in range(max(1, len(s)-2))]

def jaccard_sim(a: str, b: str) -> float:
    A = set(normalize_text_for_jaccard(a))
    B = set(normalize_text_for_jaccard(b))
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

class MemoryStore:
    """
    Memoria usando SentenceTransformers (local) para embeddings,
    ya que DeepSeek no tiene endpoint de embeddings estándar.
    """
    def __init__(self, use_embeddings: bool = True, embed_model: str = "all-MiniLM-L6-v2"):
        self.use_embeddings = use_embeddings
        self.by_cat_texts: Dict[str, List[str]] = defaultdict(list)
        self.by_cat_embs:  Dict[str, List[List[float]]] = defaultdict(list)
        
        self.encoder = None
        if self.use_embeddings:
            print(f"Cargando modelo local de embeddings: {embed_model} ...")
            self.encoder = SentenceTransformer(embed_model)

    def _embed_batch(self, texts: List[str]) -> List[List[float]]:
        if not texts or self.encoder is None:
            return []
        # Encode devuelve numpy array, convertimos a lista
        embeddings = self.encoder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        return embeddings.tolist()

    @staticmethod
    def _cos_sim(u: List[float], v: List[float]) -> float:
        import math
        num = sum(x*y for x,y in zip(u,v))
        du  = math.sqrt(sum(x*x for x in u))
        dv  = math.sqrt(sum(y*y for y in v))
        if du == 0 or dv == 0: return 0.0
        return num/(du*dv)

    def add(self, cat: str, texts: List[str]):
        if not texts: return
        self.by_cat_texts[cat].extend(texts)
        if self.use_embeddings:
            embs = self._embed_batch(texts)
            self.by_cat_embs[cat].extend(embs)

    def nearest(self, cat: str, query: str, k: int = 8) -> List[Tuple[str, float]]:
        corpus = self.by_cat_texts.get(cat, [])
        if not corpus: return []
        
        if self.use_embeddings and self.by_cat_embs.get(cat):
            q = self._embed_batch([query])[0]
            sims = [(t, self._cos_sim(q, e)) for t, e in zip(corpus, self.by_cat_embs[cat])]
        else:
            sims = [(t, jaccard_sim(query, t)) for t in corpus]
            
        sims.sort(key=lambda x: x[1], reverse=True)
        return sims[:k]

    def too_similar(self, cat: str, query: str, threshold: float) -> bool:
        if not self.by_cat_texts.get(cat): return False
        return any(sim >= threshold for _, sim in self.nearest(cat, query, k=1))

# =================== Prompts ===================
SYSTEM_PROMPT_GREEDY = (
    "You are rewriting short toxic social media posts (tweets). "
    "Given a MASKED input, you will produce new toxic tweets that keep the SAME toxicity, stance, and targets.\n"
    "STYLE RULES:\n"
    "- Write like a real toxic tweet: short, informal, sometimes slang.\n"
    "- Max 1–2 sentences per variant.\n"
    "HARD RULES:\n"
    "1) Preserve EVERY bracketed token EXACTLY as written (verbatim): [SLUR:CODE], [TARGET:CODE], [MENTION], [URL].\n"
    "2) Do NOT introduce or reveal explicit slurs; only keep the placeholders.\n"
    "3) Vary wording structure.\n"
    "5) You MUST respond with a single JSON object: {\"variants\": [\"...\"]}.\n"
)

SYSTEM_PROMPT_MEMORY = (
    "You are rewriting short toxic social media posts (tweets). "
    "Given a MASKED input, you will produce new toxic tweets that keep the SAME toxicity, stance, and targets.\n"
    "STYLE RULES:\n"
    "- Write like a real toxic tweet: short, informal, sometimes slang.\n"
    "- Max 1–2 sentences per variant.\n"
    "HARD RULES:\n"
    "1) Preserve EVERY bracketed token EXACTLY as written (verbatim): [SLUR:CODE], [TARGET:CODE], [MENTION], [URL].\n"
    "2) Do NOT introduce or reveal explicit slurs; only keep the placeholders.\n"
    "3) Vary wording and structure A LOT across variants.\n"
    "4) You MUST respond with a single JSON object of the form "
    "{\"variants\": [\"paraphrase 1\", \"paraphrase 2\", ...]} containing N_VARIANTS different paraphrases.\n"
    "5) Do not include any extra commentary, prose, or code fences outside the JSON."
)


def paraphrase_memory_multi(masked_text: str, cat: str, lang: str, mem: MemoryStore, n_variants: int) -> List[str]:
    negs = mem.nearest(cat, masked_text, k=MAX_NEG_EXAMPLES)
    neg_texts = [t for (t, s) in negs]
    neg_block = "\n---\n".join(neg_texts) if neg_texts else "(none)"

    user_prompt = (
        f"N_VARIANTS: {n_variants}\n"
        f"LANG: {lang}\n"
        f"CATEGORY: {cat}\n"
        "MASKED INPUT:\n"
        f"{masked_text}\n\n"
        "NEGATIVE EXAMPLES (avoid similar wording):\n"
        f"{neg_block}\n\n"
        "Return a JSON object: {\"variants\": [\"paraphrase 1\", \"paraphrase 2\", ...]}."
    )

    raw = call_with_retries(SYSTEM_PROMPT_MEMORY, user_prompt, max_tokens=1024) # DeepSeek context window is large
    s = _strip_fences(raw)

    variants: List[str] = []
    try:
        # Limpieza extra por si DeepSeek mete texto antes/despues del json
        json_match = re.search(r"\{.*\}", s, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group(0))
            vlist = data.get("variants", [])
            if isinstance(vlist, list):
                variants = [str(v).strip() for v in vlist if str(v).strip()]
        else:
            # Fallback a linea simple si falla JSON
            if s.strip(): variants = [s.strip()]
    except Exception:
        if s.strip(): variants = [s.strip()]

    return variants

def paraphrase_greedy(masked_text: str, cat: str, lang: str) -> str:
    user_prompt = f"LANG: {lang}\nCATEGORY: {cat}\nMASKED INPUT:\n{masked_text}\n\nOutput ONLY the JSON with 1 variant."
    raw = call_with_retries(SYSTEM_PROMPT_GREEDY, user_prompt, max_tokens=512)
    s = _strip_fences(raw)
    try:
        data = json.loads(s)
        return data["variants"][0]
    except:
        return s 

def jitter_hparams(base_temp=1.1, base_top_p=0.95):
    # DeepSeek maneja bien temperaturas ligeramente más altas
    temp = min(1.5, max(0.7, random.gauss(base_temp, 0.15)))
    top_p = min(1.0, max(0.85, random.gauss(base_top_p, 0.05)))
    return dict(temperature=temp, top_p=top_p)

# =================== Llamadas LLM ===================
def call_chat_once(system_prompt: str, user_prompt: str, max_tokens: int = 512, **kw) -> str:
    hp = jitter_hparams()
    hp.update(kw)
    
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=max_tokens,
        **hp
    )
    out = (resp.choices[0].message.content or "").strip()
    return _strip_fences(out)

def call_with_retries(system_prompt: str, user_prompt: str, retries: int = BP_RETRIES, **kw) -> str:
    backoff = 1.0
    last_e = None
    for attempt in range(retries + 1):
        try:
            out = call_chat_once(system_prompt, user_prompt, **kw)
            if not out:
                raise ValueError("Empty output")
            return out
        except Exception as e:
            last_e = e
            if attempt >= retries:
                # print(f"Error final: {e}")
                pass
            time.sleep(backoff + random.random() * 0.5)
            backoff = min(16.0, backoff * 2)
    return "" # Retornar vacio si falla todo

# =================== Núcleo Balancing ===================
def compute_targets(by_cat: Dict[str, List[Dict[str, Any]]]) -> Tuple[int, Dict[str, int]]:
    counts = {c: len(v) for c, v in by_cat.items()}
    if not counts: return 0, {}
    target_size = max(counts.values())
    needs = {c: max(0, target_size - n) for c, n in counts.items()}
    return target_size, needs

def distribute_k_per_item(n_need: int, n_items: int) -> List[int]:
    if n_need <= 0 or n_items == 0:
        return [0] * n_items
    base = n_need // n_items
    rem  = n_need % n_items
    ks = [base] * n_items
    for i in range(rem):
        ks[i] += 1
    random.shuffle(ks)
    return ks

# =================== Main ===================
def main():
    init_client()
    print(f"--- Usando API DeepSeek con modelo: {LLM_MODEL} ---")
    
    rows = [r for r in read_jsonl(INPUT_JSONL) if is_valid_row(r)]
    # Filtrar solo los que tienen tokens especiales
    rows = [r for r in rows if isinstance(r.get("text_masked"), str) and STRICT_TOKEN.search(r["text_masked"])]
    
    if not rows:
        print("No hay filas válidas con tokens [SLUR/TARGET].")
        return

    by_cat = defaultdict(list)
    for r in rows:
        by_cat[r["predicted_hate_category"]].append(r)

    target_size, needs = compute_targets(by_cat)
    print("Distribución inicial:", {c: len(v) for c, v in by_cat.items()})
    print("Objetivo por clase:", target_size)

    # Inicializar memoria (cargará SentenceTransformer localmente)
    memory = MemoryStore(use_embeddings=USE_EMBEDDINGS, embed_model=EMBED_MODEL)

    # Precarga
    for cat, items in by_cat.items():
        base_texts = [it["text_masked"] for it in items]
        base_texts = base_texts[:RAG_NEIGHBORS_PER_CAT]
        memory.add(cat, base_texts)

    augmented: List[Dict[str, Any]] = []
    ordered_cats = sorted(by_cat.keys(), key=lambda c: needs.get(c, 0), reverse=True)

    for cat in ordered_cats:
        items = by_cat[cat]
        need  = needs.get(cat, 0) + 100
        if need <= 0:
            continue

        ks = distribute_k_per_item(need, len(items))
        produced = 0
        random.shuffle(items)

        print(f"[{cat}] Iniciando aumento. Necesita: {need}")

        for row, k_i in zip(items, ks):
            if k_i <= 0: continue

            lang        = row.get("lang", "en")
            parent_id   = row.get("id")
            masked_orig = row["text_masked"]
            orig_tokens = [m.group(0) for m in STRICT_TOKEN.finditer(masked_orig)]

            # Lógica principal de Memoria
            if BP_MODE == "memory":
                remaining   = k_i
                bp_step     = 0
                masked_seed = masked_orig
                calls_used  = 0

                while remaining > 0 and calls_used < BP_MAX_CALLS_PER_ITEM:
                    calls_used += 1
                    n_variants = min(BP_VARIANTS_PER_CALL, max(1, remaining * 2))

                    cand_list = paraphrase_memory_multi(
                        masked_seed, cat, lang, memory, n_variants=n_variants
                    )
                    
                    if not cand_list: break

                    for cand in cand_list:
                        if remaining <= 0: break
                        cand = _strip_fences(cand).strip()
                        if not cand: continue

                        # Checks
                        if not contains_all(orig_tokens, cand): continue
                        if jaccard_sim(masked_orig, cand) < MIN_JACCARD_FOR_LABEL: continue
                        if memory.too_similar(cat, cand, SIM_THRESHOLD): continue
                        if jaccard_sim(masked_orig, cand) > MAX_JACCARD_FOR_DIVERSITY: continue

                        memory.add(cat, [cand])

                        new_row = dict(row)
                        bp_step += 1
                        new_row["id"] = f"{row.get('id','NA')}_bp_{uuid.uuid4().hex[:8]}"
                        new_row["text_masked"] = cand
                        new_row["augmented"] = True
                        new_row["source_id"] = parent_id
                        new_row["bp_step"] = bp_step

                        augmented.append(new_row)
                        produced  += 1
                        remaining -= 1
                        
                        print(f"  [{cat}] Generado: {cand[:60]}...")
                        masked_seed = cand
                        parent_id   = new_row["id"]

            elif BP_MODE == "greedy":
                # Lógica greedy simplificada
                masked_in = masked_orig
                for step in range(k_i):
                    cand = paraphrase_greedy(masked_in, cat, lang)
                    if cand and contains_all(orig_tokens, cand):
                        new_row = dict(row)
                        new_row["id"] = f"{row.get('id')}_gd_{uuid.uuid4().hex[:6]}"
                        new_row["text_masked"] = cand
                        new_row["augmented"] = True
                        augmented.append(new_row)
                        produced += 1
                        masked_in = cand
                        print(f"  [{cat}] Greedy: {cand[:60]}...")

        print(f"[{cat}] Finalizado. Producidos: {produced}")

    # Guardar
    out_rows = rows + augmented
    # Recorte final para balanceo exacto
    final_by_cat = defaultdict(list)
    for r in out_rows:
        final_by_cat[r["predicted_hate_category"]].append(r)

    trimmed = []
    for c, lst in final_by_cat.items():
        if len(lst) > target_size:
            trimmed.extend(lst[:target_size])
        else:
            trimmed.extend(lst)

    write_jsonl(OUTPUT_JSONL, trimmed)
    print("\nGuardado en:", OUTPUT_JSONL)
    print("Total filas:", len(trimmed))

if __name__ == "__main__":
    main()

/home/mrodrive/miniconda3/envs/hs-env-clean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Usando API DeepSeek con modelo: deepseek-chat ---
Distribución inicial: {'disability': 20, 'gender': 15, 'origin': 16, 'other': 9, 'religion': 17, 'sexual_orientation': 20}
Objetivo por clase: 20
Cargando modelo local de embeddings: all-MiniLM-L6-v2 ...


/home/mrodrive/miniconda3/envs/hs-env-clean/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


[other] Iniciando aumento. Necesita: 11


KeyboardInterrupt: 

In [1]:
import os, sys, time, json, math, random, uuid, re
import csv
from typing import List, Dict, Any, Tuple
from collections import Counter, defaultdict
from pathlib import Path

# Instalar: pip install openai sentence-transformers
from sentence_transformers import SentenceTransformer
from openai import OpenAI

SEED = 42
random.seed(SEED)

# =================== Configuración ===================
# Modelo LLM (DeepSeek)
LLM_MODEL = os.getenv("LLM_MODEL", "deepseek-chat")

# Embeddings Locales (Script 2 logic)
EMBED_MODEL = os.getenv("EMBED_MODEL", "all-MiniLM-L6-v2")

# Configuración de balanceo (Traída del Script 1 para mayor flexibilidad)
MIN_SAMPLES_PER_CLASS = int(os.getenv("MIN_SAMPLES_PER_CLASS", "200"))
FIXED_AUG_PER_ITEM    = int(os.getenv("FIXED_AUG_PER_ITEM", "0"))

# Configuración de Augmentación
BP_MODE = os.getenv("BP_MODE", "memory")  # greedy | memory
BP_VARIANTS_PER_CALL  = int(os.getenv("BP_VARIANTS_PER_CALL", "4"))
BP_MAX_CALLS_PER_ITEM = int(os.getenv("BP_MAX_CALLS_PER_ITEM", "3"))

# Filtros de calidad
MIN_JACCARD_FOR_LABEL = float(os.getenv("MIN_JACCARD_FOR_LABEL", "0.25"))
MAX_JACCARD_FOR_DIVERSITY = float(os.getenv("MAX_JACCARD_FOR_DIVERSITY", "0.92"))

# Rutas de entrada/salida
INPUT_FILE  = os.getenv("INPUT_FILE",  "en_dataset_with_stop_words/en_dataset_with_stop_words_masked_train_low_regime_augmented.jsonl") # Puede ser .csv o .jsonl
OUTPUT_JSONL = os.getenv("OUTPUT_JSONL", "en_dataset_with_stop_words/en_dataset_with_stop_words_masked_train_low_regime_augmentedv2.jsonl")

# Memoria / RAG
USE_EMBEDDINGS = os.getenv("USE_EMBEDDINGS", "1") not in ("0","false","False")
SIM_THRESHOLD  = float(os.getenv("SIM_THRESHOLD", "0.88"))
MAX_NEG_EXAMPLES = int(os.getenv("MAX_NEG_EXAMPLES", "4"))
RAG_NEIGHBORS_PER_CAT = int(os.getenv("RAG_NEIGHBORS_PER_CAT", "64"))

RETRIES = 2
BP_RETRIES = 6

# =================== Utilidades de Lectura (CSV/JSONL) ===================
def read_csv_any(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Limpieza básica de Nones
            clean = {k: (v if v is not None else "") for k, v in row.items()}
            rows.append(clean)
    return rows

def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def read_input_auto(path: str) -> List[Dict[str, Any]]:
    """Detecta automáticamente si es CSV o JSONL"""
    lower = path.lower()
    if lower.endswith(".jsonl") or lower.endswith(".jl"):
        return read_jsonl(path)
    elif lower.endswith(".csv"):
        return read_csv_any(path)
    else:
        raise ValueError(f"Formato no soportado para: {path}")

def write_jsonl(path: str, rows: List[Dict[str, Any]]):
    Path(os.path.dirname(path) or ".").mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def is_valid_row(row: Dict[str, Any]) -> bool:
    cat = row.get("predicted_hate_category", None)
    return (cat is not None) and (str(cat).lower() != "nan") and bool(row.get("text","").strip())

# =================== Cliente DeepSeek ===================
client = None

def read_api_key_from_file(path: str = "DAPI_KEY") -> str:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"ERROR: No se encontró el fichero '{path}' con la API Key.")
        sys.exit(1)

def init_client():
    global client
    if not os.getenv("DEEPSEEK_API_KEY"):
        os.environ["DEEPSEEK_API_KEY"] = read_api_key_from_file("DAPI_KEY")
    
    # Inicialización para DeepSeek
    client = OpenAI(
        api_key=os.environ["DEEPSEEK_API_KEY"], 
        base_url="https://api.deepseek.com"
    )

# =================== Regex y Tokens ===================
STRICT_TOKEN = re.compile(r"\[(SLUR|TARGET):([A-Z0-9_]+)\]")
FENCE_RE     = re.compile(r"^```(?:\w+)?\s*([\s\S]*?)\s*```$", re.IGNORECASE)

def _strip_fences(s: str) -> str:
    m = FENCE_RE.match(s.strip())
    return m.group(1).strip() if m else s.strip()

def contains_all(orig_tokens: List[str], s: str) -> bool:
    return all(tok in s for tok in orig_tokens)

# =================== Memoria (Local Embeddings) ===================
def normalize_text_for_jaccard(s: str) -> List[str]:
    s = re.sub(r"\s+", " ", s.lower()).strip()
    return [s[i:i+3] for i in range(max(1, len(s)-2))]

def jaccard_sim(a: str, b: str) -> float:
    A = set(normalize_text_for_jaccard(a))
    B = set(normalize_text_for_jaccard(b))
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

class MemoryStore:
    """Usa SentenceTransformers localmente (lógica del Script 2)"""
    def __init__(self, use_embeddings: bool = True, embed_model: str = "all-MiniLM-L6-v2"):
        self.use_embeddings = use_embeddings
        self.by_cat_texts: Dict[str, List[str]] = defaultdict(list)
        self.by_cat_embs:  Dict[str, List[List[float]]] = defaultdict(list)
        
        self.encoder = None
        if self.use_embeddings:
            print(f"Cargando modelo local de embeddings: {embed_model} ...")
            self.encoder = SentenceTransformer(embed_model)

    def _embed_batch(self, texts: List[str]) -> List[List[float]]:
        if not texts or self.encoder is None:
            return []
        embeddings = self.encoder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        return embeddings.tolist()

    @staticmethod
    def _cos_sim(u: List[float], v: List[float]) -> float:
        import math
        num = sum(x*y for x,y in zip(u,v))
        du  = math.sqrt(sum(x*x for x in u))
        dv  = math.sqrt(sum(y*y for y in v))
        if du == 0 or dv == 0: return 0.0
        return num/(du*dv)

    def add(self, cat: str, texts: List[str]):
        if not texts: return
        self.by_cat_texts[cat].extend(texts)
        if self.use_embeddings:
            embs = self._embed_batch(texts)
            self.by_cat_embs[cat].extend(embs)

    def nearest(self, cat: str, query: str, k: int = 8) -> List[Tuple[str, float]]:
        corpus = self.by_cat_texts.get(cat, [])
        if not corpus: return []
        
        if self.use_embeddings and self.by_cat_embs.get(cat):
            q = self._embed_batch([query])[0]
            sims = [(t, self._cos_sim(q, e)) for t, e in zip(corpus, self.by_cat_embs[cat])]
        else:
            sims = [(t, jaccard_sim(query, t)) for t in corpus]
            
        sims.sort(key=lambda x: x[1], reverse=True)
        return sims[:k]

    def too_similar(self, cat: str, query: str, threshold: float) -> bool:
        if not self.by_cat_texts.get(cat): return False
        return any(sim >= threshold for _, sim in self.nearest(cat, query, k=1))

# =================== Prompts (Optimizado para DeepSeek) ===================
SYSTEM_PROMPT_MEMORY = (
    "You are rewriting short toxic social media posts (tweets). "
    "Given a MASKED input, you will produce new toxic tweets that keep the SAME toxicity, stance, and targets.\n"
    "STYLE RULES:\n"
    "- Write like a real toxic tweet: short, informal, sometimes slang.\n"
    "- Max 1–2 sentences per variant.\n"
    "HARD RULES:\n"
    "1) Preserve EVERY bracketed token EXACTLY as written (verbatim): [SLUR:CODE], [TARGET:CODE], [MENTION], [URL].\n"
    "2) Do NOT introduce or reveal explicit slurs; only keep the placeholders.\n"
    "3) Vary wording and structure A LOT across variants.\n"
    "4) You MUST respond with a single JSON object of the form "
    "{\"variants\": [\"paraphrase 1\", \"paraphrase 2\", ...]} containing N_VARIANTS different paraphrases.\n"
    "5) Do not include any extra commentary, prose, or code fences outside the JSON."
)

SYSTEM_PROMPT_GREEDY = (
    "You are rewriting short toxic social media posts (tweets). "
    "Output ONLY a JSON object: {\"variants\": [\"...\"]} with 1 variant. "
    "Preserve [SLUR:CODE] and [TARGET:CODE] verbatim."
)

# =================== Lógica de llamada LLM ===================
def jitter_hparams(base_temp=1.1, base_top_p=0.95):
    temp = min(1.5, max(0.7, random.gauss(base_temp, 0.15)))
    top_p = min(1.0, max(0.85, random.gauss(base_top_p, 0.05)))
    return dict(temperature=temp, top_p=top_p)

def call_chat_once(system_prompt: str, user_prompt: str, max_tokens: int = 512, **kw) -> str:
    hp = jitter_hparams()
    hp.update(kw)
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=max_tokens,
        **hp
    )
    out = (resp.choices[0].message.content or "").strip()
    return _strip_fences(out)

def call_with_retries(system_prompt: str, user_prompt: str, retries: int = BP_RETRIES, **kw) -> str:
    backoff = 1.0
    for attempt in range(retries + 1):
        try:
            out = call_chat_once(system_prompt, user_prompt, **kw)
            if not out: raise ValueError("Empty output")
            return out
        except Exception:
            if attempt >= retries: pass
            time.sleep(backoff + random.random() * 0.5)
            backoff = min(16.0, backoff * 2)
    return ""

def paraphrase_memory_multi(masked_text: str, cat: str, lang: str, mem: MemoryStore, n_variants: int) -> List[str]:
    negs = mem.nearest(cat, masked_text, k=MAX_NEG_EXAMPLES)
    neg_texts = [t for (t, s) in negs]
    neg_block = "\n---\n".join(neg_texts) if neg_texts else "(none)"

    user_prompt = (
        f"N_VARIANTS: {n_variants}\nLANG: {lang}\nCATEGORY: {cat}\n"
        f"MASKED INPUT:\n{masked_text}\n\n"
        f"NEGATIVE EXAMPLES (avoid similar wording):\n{neg_block}\n\n"
        "Return a JSON object: {\"variants\": [\"paraphrase 1\", \"paraphrase 2\", ...]}."
    )
    
    raw = call_with_retries(SYSTEM_PROMPT_MEMORY, user_prompt, max_tokens=1024)
    s = _strip_fences(raw)
    variants = []
    try:
        # Búsqueda robusta de JSON
        json_match = re.search(r"\{.*\}", s, re.DOTALL)
        if json_match:
            data = json.loads(json_match.group(0))
            vlist = data.get("variants", [])
            if isinstance(vlist, list):
                variants = [str(v).strip() for v in vlist if str(v).strip()]
        else:
            if s.strip(): variants = [s.strip()]
    except:
        if s.strip(): variants = [s.strip()]
    return variants

def paraphrase_greedy(masked_text: str, cat: str, lang: str) -> str:
    user_prompt = f"LANG: {lang}\nCATEGORY: {cat}\nMASKED INPUT:\n{masked_text}\n"
    raw = call_with_retries(SYSTEM_PROMPT_GREEDY, user_prompt, max_tokens=512)
    try:
        data = json.loads(_strip_fences(raw))
        return data["variants"][0]
    except:
        return raw

# =================== Balanceo y Distribución ===================
def compute_targets(by_cat: Dict[str, List[Dict[str, Any]]]) -> Tuple[int, Dict[str, int]]:
    counts = {c: len(v) for c, v in by_cat.items()}
    if not counts: return 0, {}
    
    # Lógica de Script 1: Máximo entre (Clase Mayoritaria, Mínimo Absoluto)
    current_max = max(counts.values())
    target_size = max(current_max, MIN_SAMPLES_PER_CLASS)
    
    needs = {c: max(0, target_size - n) for c, n in counts.items()}
    return target_size, needs

def distribute_k_per_item(n_need: int, n_items: int) -> List[int]:
    if n_need <= 0 or n_items == 0: return [0] * n_items
    base = n_need // n_items
    rem  = n_need % n_items
    ks = [base] * n_items
    for i in range(rem): ks[i] += 1
    random.shuffle(ks)
    return ks

# =================== Main ===================
def main():
    init_client()
    print(f"--- DeepSeek Engine + CSV Support + Local Embeddings ---")
    
    # 1. Leer Entrada (CSV o JSONL)
    print(f"Leyendo: {INPUT_FILE}")
    try:
        rows = [r for r in read_input_auto(INPUT_FILE) if is_valid_row(r)]
    except Exception as e:
        print(f"Error leyendo archivo: {e}")
        return

    # 2. Filtrar filas válidas con tokens
    rows = [r for r in rows if isinstance(r.get("text_masked"), str) and STRICT_TOKEN.search(r["text_masked"])]
    if not rows:
        print("No hay filas válidas con tokens [SLUR/TARGET].")
        return

    by_cat = defaultdict(list)
    for r in rows:
        by_cat[r["predicted_hate_category"]].append(r)

    # 3. Calcular objetivos de augmentación
    if FIXED_AUG_PER_ITEM > 0:
        print(f"Modo FIJO: {FIXED_AUG_PER_ITEM} augmentaciones por fila.")
        target_size = None # No aplica objetivo global
        needs = {c: len(items) * FIXED_AUG_PER_ITEM for c, items in by_cat.items()}
    else:
        target_size, needs = compute_targets(by_cat)
        print(f"Modo BALANCEO: Objetivo por clase = {target_size} (Min Floor: {MIN_SAMPLES_PER_CLASS})")
        print("Necesidades:", needs)

    # 4. Inicializar Memoria
    memory = MemoryStore(use_embeddings=USE_EMBEDDINGS, embed_model=EMBED_MODEL)
    # Precarga
    for cat, items in by_cat.items():
        base_texts = [it["text_masked"] for it in items][:RAG_NEIGHBORS_PER_CAT]
        memory.add(cat, base_texts)

    augmented = []
    ordered_cats = sorted(by_cat.keys(), key=lambda c: needs.get(c, 0), reverse=True)

    # 5. Loop Principal
    for cat in ordered_cats:
        items = by_cat[cat]
        need = needs.get(cat, 0)
        if need <= 0: continue

        if FIXED_AUG_PER_ITEM > 0:
            ks = [FIXED_AUG_PER_ITEM] * len(items)
        else:
            ks = distribute_k_per_item(need, len(items))

        produced = 0
        random.shuffle(items)
        print(f"[{cat}] Iniciando... Necesita: {need}")

        for row, k_i in zip(items, ks):
            if k_i <= 0: continue

            lang = row.get("lang", "en")
            parent_id = row.get("id", "NA")
            masked_orig = row["text_masked"]
            orig_tokens = [m.group(0) for m in STRICT_TOKEN.finditer(masked_orig)]

            # Calcular llamadas dinámicas si necesitamos muchas variantes de un solo item
            # (Lógica rescatada del Script 1 para manejar clases muy pequeñas)
            dynamic_max_calls = max(BP_MAX_CALLS_PER_ITEM, math.ceil(k_i / BP_VARIANTS_PER_CALL) + 2)

            if BP_MODE == "memory":
                remaining = k_i
                bp_step = 0
                masked_seed = masked_orig
                calls_used = 0

                while remaining > 0 and calls_used < dynamic_max_calls:
                    calls_used += 1
                    n_variants = min(BP_VARIANTS_PER_CALL, max(1, remaining * 2))

                    cand_list = paraphrase_memory_multi(masked_seed, cat, lang, memory, n_variants)
                    
                    if not cand_list: break

                    for cand in cand_list:
                        if remaining <= 0: break
                        cand = _strip_fences(cand).strip()
                        
                        if not cand or not contains_all(orig_tokens, cand): continue
                        if jaccard_sim(masked_orig, cand) < MIN_JACCARD_FOR_LABEL: continue
                        if memory.too_similar(cat, cand, SIM_THRESHOLD): continue
                        if jaccard_sim(masked_orig, cand) > MAX_JACCARD_FOR_DIVERSITY: continue

                        # Éxito
                        memory.add(cat, [cand])
                        bp_step += 1
                        
                        new_row = dict(row)
                        new_row["id"] = f"{parent_id}_bp_{uuid.uuid4().hex[:8]}"
                        new_row["text_masked"] = cand
                        new_row["augmented"] = True
                        new_row["aug_method"] = "deepseek_memory"
                        new_row["source_id"] = parent_id
                        new_row["bp_step"] = bp_step
                        
                        augmented.append(new_row)
                        produced += 1
                        remaining -= 1
                        
                        print(f"  [{cat}] +1 (Step {bp_step}): {cand[:50]}...")
                        
                        # Teléfono roto: la salida es la nueva semilla
                        masked_seed = cand
                        parent_id = new_row["id"]

            elif BP_MODE == "greedy":
                masked_in = masked_orig
                for step in range(k_i):
                    cand = paraphrase_greedy(masked_in, cat, lang)
                    if cand and contains_all(orig_tokens, cand):
                        new_row = dict(row)
                        new_row["id"] = f"{parent_id}_gd_{uuid.uuid4().hex[:6]}"
                        new_row["text_masked"] = cand
                        new_row["augmented"] = True
                        augmented.append(new_row)
                        produced += 1
                        masked_in = cand
                        print(f"  [{cat}] Greedy: {cand[:50]}...")

    # 6. Guardado Final
    out_rows = rows + augmented
    
    # Recorte si no estamos en modo fijo
    if FIXED_AUG_PER_ITEM == 0 and target_size:
        final_by_cat = defaultdict(list)
        for r in out_rows:
            final_by_cat[r["predicted_hate_category"]].append(r)
        
        trimmed = []
        for c, lst in final_by_cat.items():
            if len(lst) > target_size:
                trimmed.extend(lst[:target_size])
            else:
                trimmed.extend(lst)
        out_rows = trimmed

    write_jsonl(OUTPUT_JSONL, out_rows)
    print(f"\nProceso completado. Guardado en: {OUTPUT_JSONL}")
    print(f"Total filas finales: {len(out_rows)}")

if __name__ == "__main__":
    main()

/home/mrodrive/miniconda3/envs/hs-env-clean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- DeepSeek Engine + CSV Support + Local Embeddings ---
Leyendo: en_dataset_with_stop_words/en_dataset_with_stop_words_masked_train_low_regime_augmented.jsonl
Modo BALANCEO: Objetivo por clase = 200 (Min Floor: 200)
Necesidades: {'disability': 180, 'gender': 185, 'origin': 184, 'other': 191, 'religion': 183, 'sexual_orientation': 180}
Cargando modelo local de embeddings: all-MiniLM-L6-v2 ...


AcceleratorError: CUDA error: CUDA-capable device(s) is/are busy or unavailable
Search for `cudaErrorDevicesUnavailable' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
